# SupplyMind AI — Feature Selection

Practice the same statistical-selection ideas as the reference notebook while avoiding purely automatic feature dropping.

Production selection follows:

1. availability/leakage,
2. identifiers/cardinality,
3. statistical association,
4. multicollinearity/redundancy,
5. model importance after training.

In [ ]:
# -------------------
# Imports
# -------------------

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from supplymind.features.predictions.domain.constants import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
    TARGET_COLUMN,
)
from supplymind.features.predictions.ml.workflow import load_clean_syndelay

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
# -------------------
# Project configuration
# -------------------

DATASET_PATH = Path("../data/raw/syndelay/syndelay_v1.csv")
REPORT_ROOT = Path("../reports")

assert DATASET_PATH.exists(), (
    f"Dataset not found: {DATASET_PATH}"
)

In [ ]:
from supplymind.features.predictions.ml.features import (
    engineer_features,
    feature_groups,
)
from supplymind.features.predictions.ml.feature_selection import (
    chi_square_categorical_report,
    mutual_information_report,
    numeric_correlation_report,
)

df = load_clean_syndelay(DATASET_PATH)
featured = engineer_features(df)
numerical_features, categorical_features = feature_groups()

## 1. Categorical selection — Chi-square

In [ ]:
categorical_report = chi_square_categorical_report(
    featured,
    categorical_features,
    TARGET_COLUMN,
)
categorical_report

,feature,chi2,p_value,degrees_of_freedom,significant_at_0_05
0,shipping_mode,34852.148947,0.000000,3,True
1,customer_state,54.427110,0.113547,43,False
2,payment_type,5.054934,0.167818,3,False
3,order_region,27.271105,0.201110,22,False
4,category_name,55.276213,0.249678,49,False
5,department_name,12.071760,0.280283,10,False
6,order_country,168.522941,0.306654,160,False
7,customer_segment,1.845558,0.397413,2,False
8,market,3.752437,0.440545,4,False
9,product_name,115.663869,0.517564,117,False


### Conclusion

Only `shipping_mode` is significant at the 0.05 level here; every other categorical
feature has p > 0.05. I am not dropping the rest automatically because significance is
only one view of usefulness, and tree models may still use interactions between otherwise
weak variables.

## 2. Numerical selection — correlation

In [ ]:
numeric_corr = numeric_correlation_report(
    featured,
    numerical_features,
    TARGET_COLUMN,
)
numeric_corr

,feature,correlation
0,profit_per_order,-0.008580
1,order_profit_per_order,-0.006619
2,order_hour,0.006173
3,order_year,0.003419
4,order_item_quantity,-0.003071
5,order_item_profit_ratio,-0.003047
6,order_item_discount_rate,0.003016
7,order_day,-0.002937
8,longitude,-0.002746
9,sales,-0.002589


### Conclusion

The numerical correlations are effectively zero. The largest absolute value is only
about **0.0086** (`profit_per_order`). This is a strong reason not to rely on a purely
linear feature-selection rule.

## 3. Numerical selection — mutual information

In [ ]:
numeric_mi = mutual_information_report(
    featured,
    numerical_features,
    TARGET_COLUMN,
)
numeric_mi

,feature,mutual_information
0,customer_city_frequency,0.009518
1,order_year,0.006617
2,order_quarter,0.004647
3,order_item_quantity,0.003765
4,order_state_frequency,0.003275
5,order_weekday,0.003261
6,order_is_weekend,0.002434
7,product_price,0.002315
8,latitude,0.002278
9,sales,0.002217


### Conclusion

The strongest numerical mutual-information score is only **0.0095** for
`customer_city_frequency`; `order_year` follows at **0.0066**. I therefore treat the
numeric fields as weak supporting signals rather than individually predictive features.

## 4. Low-variance inspection

In [ ]:
variance = (
    featured[numerical_features]
    .var(numeric_only=True)
    .sort_values()
    .rename("variance")
    .to_frame()
)

variance

,variance
order_city_frequency,0.000008
order_state_frequency,0.000082
order_item_discount_rate,0.005028
customer_city_frequency,0.031873
order_is_weekend,0.203804
order_item_profit_ratio,0.2092
order_year,0.678368
order_quarter,1.223316
order_item_quantity,2.089396
order_weekday,3.987036


### Conclusion

The frequency-encoded location fields naturally have small variance because they are
proportions, so I will not remove them using a raw variance threshold. The equal variance
of `product_price` and `order_item_product_price` does, however, reinforce that I should
watch for redundant economic features during model-importance review.

## 5. Final selection rationale

## Selection decision

I am keeping the leakage-safe feature contract for the first model comparison rather than
forcing an aggressive statistical cut. The EDA already shows that most individual signals
are weak and that `shipping_mode` is exceptional. The model comparison and feature
importance will tell me whether the remaining features improve generalization enough to
justify keeping them.